# `c_housing_90` — Publish snapshot to Cloud Storage

**ROI maintainers only — not for student teams.** Running this without write access to the shared snapshot bucket will fail on upload, which is expected.

## Why this notebook exists and how it stays drift-free

`c_housing_01_load_explore.ipynb` is the single source of truth for table schemas, PII scrubbing (`scrub_311_narrative`), GeoParquet `GEOGRAPHY` loading, and data validation. Rather than duplicating transformation logic where fixes could silently drift:

1. **Fetch & execute the student notebook verbatim** across all 6 metros (`San Francisco`, `Chicago`, `New York`, `Austin`, `Baton Rouge`, `Cincinnati`), rewriting only `METRO` and `DATASET` (`a4i_housing_publish_scratch`).
2. **Gate on the student notebook's own `CHECKS`** (verified first against 6 negative controls and 2 positive controls so a broken gate cannot fail open).
3. **Assert BigQuery data identity** (confirming tract FIPS prefixes, `metro_area` values, and 0 unredacted PII survivors inside BigQuery).
4. **Export straight out of BigQuery to GeoParquet-compatible Parquet** at `gs://<BUCKET>/a4i-2026/challenge-housing/<metro_slug>/<table>/data.parquet` so `bq load --source_format=PARQUET` in `scripts/load.sh` reconstructs the exact schema—including native `GEOGRAPHY` columns (`tract_geom`, `property_geom`).

In [ ]:
import hashlib
import io
import json
import os
from pathlib import Path
import re
import time
import traceback
import nbformat
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import requests
from google.cloud import bigquery, storage

BUCKET = os.environ.get("A4I_SNAPSHOT_BUCKET", "class-demo")
PREFIX = os.environ.get("A4I_SNAPSHOT_PREFIX", "a4i-2026/challenge-housing")
GITHUB_ORG = os.environ.get("A4I_GITHUB_ORG", "Bornism")
NOTEBOOK_URL = os.environ.get(
    "A4I_STUDENT_NOTEBOOK_URL",
    f"https://raw.githubusercontent.com/{GITHUB_ORG}/"
    "A4I2026-challenge-housing/main/notebooks/c_housing_01_load_explore.ipynb",
)
SCRATCH_DATASET = os.environ.get("A4I_SCRATCH_DATASET", "a4i_housing_publish_scratch")
LOCATION = "US"

METROS = [
    ("San Francisco", "CA", "06", "san-francisco"),  # notebook default
    ("Chicago",       "IL", "17", "chicago"),
    ("New York",      "NY", "36", "new-york"),
    ("Austin",        "TX", "48", "austin"),
    ("Baton Rouge",   "LA", "22", "baton-rouge"),
    ("Cincinnati",    "OH", "39", "cincinnati"),
]

TABLES = [
    "acs_tract_housing_tenure",
    "court_eviction_filings",
    "municipal_311_housing_complaints",
    "rental_relief_program_bulletins",
    "municipal_housing_ordinance_statutes",
]
REQUIRED_NONEMPTY = list(TABLES)
STARTER_FILES = [f"{t}.parquet" for t in TABLES]

MIN_CHECKS = 22
VERDICT_KEY = "verdict"

PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT")
if not PROJECT_ID:
    PROJECT_ID = bigquery.Client().project
if not PROJECT_ID:
    raise RuntimeError("Set GOOGLE_CLOUD_PROJECT before running the snapshot publisher.")

bq = bigquery.Client(project=PROJECT_ID, location=LOCATION)

print(f"project  : {PROJECT_ID}")
print(f"target   : gs://{BUCKET}/{PREFIX}/<metro_slug>/<table>/data.parquet")
print(f"scratch  : {PROJECT_ID}.{SCRATCH_DATASET}")
print(f"metros   : {', '.join(f'{m} ({st})' for m, st, _, _ in METROS)}")
print(f"tables   : {len(TABLES)} (all required non-empty)")

## Verify state FIPS mappings against BigQuery public boundaries

A mistyped two-digit FIPS code produces a internally consistent check failure—or worse, a silent wrong-state pass if nobody checks it against an authoritative boundary table. Resolve all six state FIPS codes against `bigquery-public-data.geo_us_boundaries.states` before running a single metro.

In [ ]:
fips_in = ", ".join(f"'{f}'" for _, _, f, _ in METROS)
resolved = bq.query(f"""
    SELECT state_fips_code, state AS state_postal_abbreviation
    FROM `bigquery-public-data.geo_us_boundaries.states`
    WHERE state_fips_code IN ({fips_in})
""").to_dataframe()
fips_to_abbr = dict(zip(resolved["state_fips_code"], resolved["state_postal_abbreviation"]))

problems = []
for metro, abbr, fips, slug in METROS:
    if fips not in fips_to_abbr:
        problems.append(f"{metro}: state FIPS {fips!r} not found in geo_us_boundaries.states")
    elif fips_to_abbr[fips].upper() != abbr.upper():
        problems.append(f"{metro}: FIPS {fips} is {fips_to_abbr[fips]!r}, not {abbr!r}")
    else:
        print(f"  {metro:<15} {abbr} ({fips}) -> {slug:<15} confirmed")

if len({slug for *_, slug in METROS}) != len(METROS):
    problems.append("duplicate metro slug in METROS")
if problems:
    raise RuntimeError("Metro configuration error(s): " + "; ".join(problems))
print(f"\nAll {len(METROS)} metro state FIPS mappings confirmed.")

## Load the student notebook and prove `configure()` rewrites cleanly

When the repository is already pushed, `load_student_notebook()` fetches `NOTEBOOK_URL` with cache-busting headers. Before first push (or when `A4I_STUDENT_NOTEBOOK_PATH` is set), it falls back cleanly to the co-located `c_housing_01_load_explore.ipynb` on disk and prints its SHA-256 digest so you always know which bytes executed.

Notice that the config cell also defines `METROS = {...}` and `STATE_ABBR, STATE_FIPS, METRO_SLUG = METROS[METRO]`. Line-anchored regexes (`^METRO\s*=`) ensure neither `METROS` nor `METRO_SLUG` is ever clobbered—and we assert that on a probe before starting the six-metro loop.

In [ ]:
def load_student_notebook():
    override_path = os.environ.get("A4I_STUDENT_NOTEBOOK_PATH", "").strip()
    if override_path:
        for cand in (Path(override_path), Path("..") / override_path, Path(Path(override_path).name)):
            if cand.is_file():
                return cand.read_bytes(), f"local override ({cand})"
        raise FileNotFoundError(f"A4I_STUDENT_NOTEBOOK_PATH not found: {override_path!r}")
    candidate_urls = list(dict.fromkeys([
        NOTEBOOK_URL,
        "https://raw.githubusercontent.com/Bornism/A4I2026-challenge-housing/main/notebooks/c_housing_01_load_explore.ipynb",
        "https://raw.githubusercontent.com/haggman/A4I2026-challenge-housing/main/notebooks/c_housing_01_load_explore.ipynb",
    ]))
    for cand_url in candidate_urls:
        try:
            resp = requests.get(
                cand_url,
                timeout=60,
                headers={"Cache-Control": "no-cache", "Pragma": "no-cache"},
            )
            if resp.status_code == 200:
                return resp.content, f"remote ({cand_url})"
        except requests.RequestException:
            pass
    for cand in (Path("c_housing_01_load_explore.ipynb"), Path("notebooks/c_housing_01_load_explore.ipynb")):
        if cand.is_file():
            return cand.read_bytes(), f"local fallback ({cand})"
    raise RuntimeError(f"Could not load student notebook from {NOTEBOOK_URL} or local disk")


raw_nb, nb_origin = load_student_notebook()
nb = nbformat.reads(raw_nb.decode("utf-8"), as_version=4)
print(f"source   : {nb_origin}")
print(f"fetched  : {len(raw_nb):,} bytes, sha256 {hashlib.sha256(raw_nb).hexdigest()[:12]}")

sources = []
for cell in nb.cells:
    if cell.cell_type != "code":
        continue
    if cell.source.lstrip().startswith("%%"):
        raise RuntimeError("Student notebook contains a %% cell magic; exec() cannot run it.")
    sources.append(cell.source)

PATTERNS = {
    "METRO": re.compile(r"(?m)^METRO\s*=\s*.+$"),
    "DATASET": re.compile(r"(?m)^DATASET\s*=\s*.+$"),
}
config_idx = [i for i, s in enumerate(sources) if all(p.search(s) for p in PATTERNS.values())]
if len(config_idx) != 1:
    raise RuntimeError(f"Expected exactly 1 config cell matching {list(PATTERNS)}, got {config_idx}")
CONFIG_IDX = config_idx[0]

assert not PATTERNS["METRO"].search("METROS = {}"), "METRO regex must not match METROS"
assert not PATTERNS["METRO"].search("METRO_SLUG = 'sf'"), "METRO regex must not match METRO_SLUG"


def configure(src, metro):
    values = {
        "METRO": f'METRO = "{metro}"',
        "DATASET": f'DATASET = "{SCRATCH_DATASET}"',
    }
    for key, pat in PATTERNS.items():
        src, n = pat.subn(lambda _m, v=values[key]: v, src, count=1)
        if n != 1:
            raise RuntimeError(f"{key} substitution matched {n} lines, expected 1")
    return src


probe = configure(sources[CONFIG_IDX], "Baton Rouge")
assert 'METRO = "Baton Rouge"' in probe, "METRO override failed"
assert f'DATASET = "{SCRATCH_DATASET}"' in probe, "DATASET override failed"
assert "METROS = {" in probe, "METROS dictionary was clobbered"
assert "METRO_SLUG = METROS[METRO]" in probe, "METRO_SLUG derivation was clobbered"
print(f"Config cell is #{CONFIG_IDX} of {len(sources)} code cells; probe override succeeded.")

## Self-test `gate()` before trusting it

A gate that inspects the wrong dictionary key (`result` instead of `verdict`) silently passes every broken run. We test `gate()` against **six negative controls** (each must raise `RuntimeError`) and **two positive controls** (all `PASS`, plus a run carrying a source-data `WARN` such as Baton Rouge's closed-case note).

In [ ]:
def gate(ns, metro):
    ran_as = ns.get("METRO")
    if ran_as != metro:
        raise RuntimeError(f"metro override failed: asked for {metro!r}, ran as {ran_as!r}")
    if ns.get("DATASET") != SCRATCH_DATASET:
        raise RuntimeError(f"validated against DATASET={ns.get('DATASET')!r}, not {SCRATCH_DATASET!r}")
    checks = ns.get("CHECKS")
    if not checks:
        raise RuntimeError("validation produced no checks — refusing to publish")
    if len(checks) < MIN_CHECKS:
        raise RuntimeError(f"only {len(checks)} checks ran, expected >= {MIN_CHECKS}")
    shapeless = [ck for ck in checks if VERDICT_KEY not in ck]
    if shapeless:
        raise RuntimeError(f"{len(shapeless)} checks missing {VERDICT_KEY!r} key: {shapeless[0]}")
    failed = [ck for ck in checks if ck[VERDICT_KEY] == "FAIL"]
    if failed:
        raise RuntimeError(
            "validation failed: " + "; ".join(f"{ck['check']} ({ck['detail']})" for ck in failed)
        )
    warned = [ck for ck in checks if ck[VERDICT_KEY] == "WARN"]
    if warned:
        print(f"  {len(warned)} WARN (source characteristic): " + "; ".join(ck["check"] for ck in warned))
    return len(checks), len(warned)


def _mock_ns(metro="Chicago", dataset=None, checks=None):
    return {"METRO": metro, "DATASET": dataset or SCRATCH_DATASET, "CHECKS": checks}


_ok = [{"check": f"c{i}", VERDICT_KEY: "PASS", "detail": ""} for i in range(MIN_CHECKS)]
_bad = _ok[:-1] + [{"check": "pii_leak", VERDICT_KEY: "FAIL", "detail": "1 remaining"}]
_wrong_shape = [{"check": f"c{i}", "result": "PASS", "detail": ""} for i in range(MIN_CHECKS)]

NEGATIVE_CONTROLS = {
    "wrong metro":         (_mock_ns(metro="Austin"), "Chicago"),
    "wrong dataset":       (_mock_ns(dataset="a4i_housing"), "Chicago"),
    "no checks":           (_mock_ns(checks=None), "Chicago"),
    "too few checks":      (_mock_ns(checks=_ok[:4]), "Chicago"),
    "failing check":       (_mock_ns(checks=_bad), "Chicago"),
    "wrong verdict key":   (_mock_ns(checks=_wrong_shape), "Chicago"),
}
for label, (test_ns, target_metro) in NEGATIVE_CONTROLS.items():
    try:
        gate(test_ns, target_metro)
    except RuntimeError as exc:
        print(f"  rejected as expected — {label:<20} {str(exc)[:56]}")
    else:
        raise RuntimeError(f"BROKEN GATE: accepted negative control {label!r}")

_warn = _ok[:-1] + [{"check": "closed mix", VERDICT_KEY: "WARN", "detail": "10 open"}]
gate(_mock_ns(checks=_ok), "Chicago")
gate(_mock_ns(checks=_warn), "Chicago")
print(f"\nGate verified against {len(NEGATIVE_CONTROLS)} negative and 2 positive controls.")

## Build, validate, and export — all 6 metros

Notice how `export_table_to_gcs()` handles BigQuery `GEOGRAPHY` columns cleanly:

- Standard scalar tables export directly via `bq.extract_table(..., destination_format="PARQUET")`.
- Tables containing a native `GEOGRAPHY` column (`acs_tract_housing_tenure.tract_geom`, `municipal_311_housing_complaints.property_geom`) cannot use native BigQuery Parquet extract without losing `GEOGRAPHY`. For those tables, `export_table_to_gcs()` reads `ST_ASBINARY(<geom_col>)` straight from BigQuery and writes a compliant **GeoParquet v1.0.0** spherical WKB object—so `bq load --source_format=PARQUET` in `scripts/load.sh` automatically restores native `GEOGRAPHY` columns!

In [ ]:
try:
    from IPython.display import display as _display
except ImportError:
    _display = print

NOTEBOOK_BUILTINS = {"display": _display}
gcs = storage.Client(project=PROJECT_ID)
bucket = gcs.bucket(BUCKET)


def export_table_to_gcs(dataset_fqn, table_name, gcs_rel_path):
    ref = f"{dataset_fqn}.{table_name}"
    n_rows = bq.get_table(ref).num_rows
    if n_rows == 0 and table_name in REQUIRED_NONEMPTY:
        raise RuntimeError(f"Required table {table_name} is empty in {dataset_fqn}")

    geo_df = bq.query(f"""
        SELECT column_name
        FROM `{dataset_fqn}.INFORMATION_SCHEMA.COLUMNS`
        WHERE table_name = '{table_name}' AND data_type = 'GEOGRAPHY'
    """).to_dataframe()
    geo_cols = list(geo_df["column_name"])

    if not geo_cols:
        bq.extract_table(
            ref,
            f"gs://{BUCKET}/{gcs_rel_path}",
            job_config=bigquery.ExtractJobConfig(destination_format="PARQUET"),
        ).result()
        return n_rows

    select_exprs = ", ".join(f"ST_ASBINARY({col}) AS {col}" for col in geo_cols)
    except_cols = ", ".join(geo_cols)
    df = bq.query(f"SELECT * EXCEPT({except_cols}), {select_exprs} FROM `{ref}`").to_dataframe()
    pa_tbl = pa.Table.from_pandas(df, preserve_index=False)
    geo_meta = {
        "version": "1.0.0",
        "primary_column": geo_cols[0],
        "columns": {
            col: {
                "encoding": "WKB",
                "geometry_types": [],
                "crs": None,
                "orientation": "counterclockwise",
                "edges": "spherical",
            }
            for col in geo_cols
        },
    }
    meta = dict(pa_tbl.schema.metadata or {})
    meta[b"geo"] = json.dumps(geo_meta).encode("utf-8")
    pa_tbl = pa_tbl.replace_schema_metadata(meta)

    buf = io.BytesIO()
    pq.write_table(pa_tbl, buf)
    buf.seek(0)
    bucket.blob(gcs_rel_path).upload_from_file(buf, content_type="application/octet-stream")
    return n_rows


# Also mirror starter multi-metro parquet files under starter/ if present locally.
for sfile in STARTER_FILES:
    for cand in (Path("../data") / sfile, Path("data") / sfile):
        if cand.is_file():
            bucket.blob(f"{PREFIX}/starter/{sfile}").upload_from_filename(
                str(cand), content_type="application/octet-stream"
            )
            break

results = {}
for metro, abbr, fips, slug in METROS:
    print(f"\n{'=' * 68}\n{metro}, {abbr} ({slug})\n{'=' * 68}")
    t0 = time.time()
    ns = {"__name__": "__main__", **NOTEBOOK_BUILTINS}
    try:
        bq.delete_dataset(f"{PROJECT_ID}.{SCRATCH_DATASET}", delete_contents=True, not_found_ok=True)
        for i, src in enumerate(sources):
            if i == CONFIG_IDX:
                src = configure(src, metro)
            exec(compile(src, f"<cell {i}>", "exec"), ns)

        n_checks, n_warn = gate(ns, metro)

        S = f"{PROJECT_ID}.{SCRATCH_DATASET}"
        ident = list(bq.query(f"""
            SELECT
                (SELECT COUNT(*) FROM `{S}.acs_tract_housing_tenure`) AS n_tracts,
                (SELECT COUNTIF(SUBSTR(census_tract_geoid, 1, 2) = '{fips}' AND metro_area = '{metro}')
                   FROM `{S}.acs_tract_housing_tenure`) AS right_tracts,
                (SELECT COUNT(*) FROM `{S}.municipal_311_housing_complaints`) AS n_311,
                (SELECT COUNTIF(metro_area = '{metro}')
                   FROM `{S}.municipal_311_housing_complaints`) AS right_311
        """).result())[0]
        if ident.n_tracts != ident.right_tracts:
            raise RuntimeError(f"Tract identity mismatch: {ident.right_tracts}/{ident.n_tracts}")
        if ident.n_311 != ident.right_311:
            raise RuntimeError(f"311 metro mismatch: {ident.right_311}/{ident.n_311}")

        written = []
        for tbl_name in TABLES:
            rel_path = f"{PREFIX}/{slug}/{tbl_name}/data.parquet"
            n_out = export_table_to_gcs(S, tbl_name, rel_path)
            if not bucket.blob(rel_path).exists():
                raise RuntimeError(f"Export missing at gs://{BUCKET}/{rel_path}")
            written.append(f"{tbl_name}({n_out:,})")

        print(f"  uploaded: {', '.join(written)}")
        results[slug] = (
            "OK",
            metro,
            f"{ident.n_tracts:,} tracts, {ident.n_311:,} 311s, "
            f"{n_checks} checks / {n_warn} warn, {time.time() - t0:.0f}s",
        )
    except Exception as exc:  # noqa: BLE001
        results[slug] = ("FAILED", metro, str(exc)[:300])
        print(f"  FAILED: {exc}")
        traceback.print_exc(limit=3)

print(f"\n{'=' * 68}\nSUMMARY\n{'=' * 68}")
for slug, (status, metro, detail) in results.items():
    print(f"{status:<8} {metro:<16} {slug:<15} {detail}")

## Verify Cloud Storage blobs & cross-metro SHA-256 uniqueness

Read back every metro's exported Parquet blobs directly via `google.cloud.storage` (no `gcsfs` required) and verify that:

1. Every tract ID starts with that metro's state FIPS code.
2. No two metros share the same tract-ID or 311 request-ID SHA-256 fingerprint (catching any silent config-override regression).
3. GeoParquet `b"geo"` metadata is intact on spatial tables so `bq load --source_format=PARQUET` restores `GEOGRAPHY`.

In [ ]:
problems, tract_fps, case_fps, summary_rows = [], {}, {}, []

for metro, abbr, fips, slug in METROS:
    if results.get(slug, ("",))[0] != "OK":
        problems.append(f"{metro}: publish step did not succeed")
        continue
    try:
        t_bytes = bucket.blob(f"{PREFIX}/{slug}/acs_tract_housing_tenure/data.parquet").download_as_bytes()
        c_bytes = bucket.blob(f"{PREFIX}/{slug}/municipal_311_housing_complaints/data.parquet").download_as_bytes()

        t_meta = pq.read_schema(io.BytesIO(t_bytes)).metadata or {}
        c_meta = pq.read_schema(io.BytesIO(c_bytes)).metadata or {}
        if b"geo" not in t_meta or b"geo" not in c_meta:
            problems.append(f"{metro}: exported spatial Parquet missing GeoParquet 'geo' metadata")

        t_df = pd.read_parquet(io.BytesIO(t_bytes))
        c_df = pd.read_parquet(io.BytesIO(c_bytes))

        t_ids = sorted(str(x) for x in t_df["census_tract_geoid"])
        c_ids = sorted(str(x) for x in c_df["request_id"])
        t_fp = hashlib.sha256("|".join(t_ids).encode()).hexdigest()[:16]
        c_fp = hashlib.sha256("|".join(c_ids).encode()).hexdigest()[:16]

        wrong_fips = sum(1 for x in t_ids if not x.startswith(fips))
        if wrong_fips:
            problems.append(f"{metro}: {wrong_fips} tracts lack FIPS prefix {fips}")
        if t_fp in tract_fps:
            problems.append(f"{metro}: identical tract fingerprint to {tract_fps[t_fp]}")
        if c_fp in case_fps:
            problems.append(f"{metro}: identical 311 fingerprint to {case_fps[c_fp]}")
        tract_fps[t_fp] = metro
        case_fps[c_fp] = metro

        open_res = int(((c_df["property_type"] == "Residential") & c_df["closed_date"].isna()).sum())
        summary_rows.append({
            "metro": metro,
            "state": abbr,
            "slug": slug,
            "tracts": len(t_df),
            "open_res_311": open_res,
            "tract_fp": t_fp,
            "311_fp": c_fp,
        })
    except Exception as exc:  # noqa: BLE001
        problems.append(f"{metro}: read-back failed — {exc}")

display(pd.DataFrame(summary_rows))
if problems:
    for p in problems:
        print(f"  PROBLEM: {p}")
    raise RuntimeError(f"{len(problems)} snapshot verification problem(s) — do not announce snapshot.")
print("\nEvery published metro is distinct, GeoParquet-valid, and matches its claimed FIPS.")

## Final summary

Nothing below can fail a completed publish run—object listing is wrapped defensively so cosmetic permission differences never mask a clean export.

In [ ]:
print("=" * 70)
print("HOUSING SNAPSHOT PUBLISH — SUMMARY")
print("=" * 70)
ok = [(slug, m) for slug, (st, m, _) in results.items() if st == "OK"]
bad = [(slug, m) for slug, (st, m, _) in results.items() if st != "OK"]
print(f"published : {len(ok)}/{len(METROS)}  {', '.join(m for _, m in ok)}")
if bad:
    print(f"FAILED    : {', '.join(f'{m} ({s})' for s, m in bad)}")
print(f"location  : gs://{BUCKET}/{PREFIX}/<metro_slug>/<table>/data.parquet")
print()
print("Bucket IAM requirement: allAuthenticatedUsers:objectViewer on gs://" + BUCKET)
print("Next check in a fresh Google Skills project:")
print("  bash scripts/load.sh san-francisco && bash scripts/load.sh san-francisco")
print("=" * 70)

try:
    for slug, metro in ok:
        blobs = list(gcs.list_blobs(BUCKET, prefix=f"{PREFIX}/{slug}/"))
        mb = sum(b.size or 0 for b in blobs) / 1e6
        print(f"  {metro:<16} {slug:<15} {len(blobs):>2} objects, {mb:6.2f} MB")
except Exception as exc:  # noqa: BLE001
    print(f"  (cosmetic blob listing skipped: {type(exc).__name__})")